
# Chapter 18: Pydantic - Based Structured Outputs in JSON
We don't need JSON just because an LLM is involved. We use JSON when we want the LLM's output to be consumed by another program reliably.



Let's build a small Transaction Risk Extractor

In [1]:
from dotenv import load_dotenv
import os

load_dotenv()


os.environ["GROQ_API_KEY"] = "YOUR API"


In [2]:
# Setting up environment variable

def setup_environment():
  """ Loads environment variables and check for the required API key"""

  load_dotenv()
  if not os.getenv("GROQ_API_KEY"):
    raise ValueError("GROQ_API_KEY is not set")


In [3]:
%pip install -U langchain langchain-core langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.7/164.7 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.1/572.1 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 7.4 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.6.3
    Uninstalling langchain-core-1.6.3:
      Successfully uninstalled langchain-core-1.6.3
  Attempting uninstall: langchain
    Found existing installation: langchain 1.4.0
    Uninstalling langchain-1.4.0:
      Successfully uninstalled langchain-1.4.0


In [4]:
!pip install -U langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 kB 3.3 MB/s eta 0:00:00


In [5]:
from langchain_openai import ChatOpenAI
import os

llm = ChatOpenAI(
    model="openai/gpt-oss-20b",
    base_url="https://api.groq.com/openai/v1",
    api_key=os.environ["GROQ_API_KEY"],

)

In [6]:
model = "openai/gpt-oss-20b"

In [11]:
from pydantic import BaseModel, Field

In [12]:
from typing import Literal


class TransactionRisk(BaseModel):

    amount: float = Field(description="Transaction amount")

    currency: str = Field(description="Transaction currency")

    source_country: str

    destination_country: str

    purpose: str

    risk_level: Literal["LOW", "MEDIUM", "HIGH"]

In [13]:
structured_llm = llm.with_structured_output(TransactionRisk)

In [14]:
transaction = """Transaction of USD 50,000 was sent from India to the UAE.
The customer has been active for 5 years and the payment
purpose is consulting services.
"""

In [15]:
response = structured_llm.invoke(
    f"""
    Extract the transaction information.

    Transaction:
    {transaction}
    """
)


In [16]:
response

TransactionRisk(amount=50000.0, currency='USD', source_country='India', destination_country='UAE', purpose='consulting services', risk_level='MEDIUM')

In [19]:
print(response)

amount=50000.0 currency='USD' source_country='India' destination_country='UAE' purpose='consulting services' risk_level='MEDIUM'


In [23]:
print(response.model_dump_json(indent = 2))

{
  "amount": 50000.0,
  "currency": "USD",
  "source_country": "India",
  "destination_country": "UAE",
  "purpose": "consulting services",
  "risk_level": "MEDIUM"
}


In [24]:
# We want LLM to give output in specific format

In [ ]:
--- THE END ----